## Langchain으로 RAG 시작하기

### step 0 : 설치와 준비

#### langchain 설치 및 openAI API키 등록

In [ ]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters

In [2]:
import os
os.environ['OPENAI_API_KEY'] = ""

In [3]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(model="gpt-4o",
                 temperature=0.0)

In [ ]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

LangChain 자체만으로는 PDF나 워드 파일 등의 내용을 직접 해석할 수 없습니다. 따라서 파일을 읽고 텍스트로 변환하는 작업을 수행해 줄 전용 도구들(pypdf, docx2txt, unstructured 등)을 시스템에 미리 설치하는 과정입니다. LangChain의 문서 불러오기(Document Loader) 기능이 이 도구들의 기능에 의존하고 있기 때문입니다.

### step 1 : Document Loaders 사용해보기

Document Loader는 RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다. https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용

In [14]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./Demian.pdf")
pages = loader.load_and_split()

In [15]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': './Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [16]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader 사용하기

In [17]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("./titanic.csv")

data = loader.load()

In [18]:
data[:3]

[Document(metadata={'source': './titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': './titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': './titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더

In [21]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

In [23]:
# print(documents[0].page_content)

### step 2 : TextSplitters 사용해보기

Text Splitter는 긴 텍스트 문서를 의미를 유지한 작은 단위(Chunk) 로 분할하는 역할을 합니다.

In [27]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

with open("./state_of_the_union.txt") as f:
    text = f.read()
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

In [28]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이 확이

In [29]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


#### 토큰 단위로 텍스트 분할해보기

In [ ]:
!pip install tiktoken

In [34]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [35]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


### step 3 : TextEmbedding 사용해보기

In [36]:
import openai
client = openai.OpenAI()

for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [37]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [38]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

In [ ]:
print(embeddings[1])

In [40]:
len(embeddings[1])

1536

##### 임베딩끼리 유사도 계산

In [42]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [43]:
query = ["this is red fruit"]

In [45]:
e_query = embedding_model.embed_documents(query)

print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7478929665954975
0.4898791411166917
0.4083791543048118


임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.
해당 링크에서 여러 목록을 확인하실 수 있습니다.
https://python.langchain.com/docs/integrations/text_embedding/

### step 4 VectorStore 사용해보기

VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다. 이 저장소는 단순한 데이터 보관 공간이 아니라, 벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

대표적인 VectorStore로는 Chroma, FAISS 등이 있으며, 각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

In [ ]:
!pip install chromadb
!pip install langchain-chroma

In [ ]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

In [49]:
from langchain_chroma import Chroma

# 위에서 사용했던 코드입니다
loader = PyPDFLoader("./Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

Chroma에 임베딩 시킵니다

In [50]:
db = Chroma.from_documents(docs, embedding_model)

query 입력

In [51]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [52]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

### step 5 : Retriever 사용해보기

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤, VectorStore에 저장된 문서 벡터들과 비교하여 의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

In [ ]:
!pip install -U langchain langchain-classic

In [54]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

In [55]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다. 구조가 단순하고 이해하기 쉬워, RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다. 단점으로는 문서 수가 많아질 경우 토큰 사용량이 빠르게 증가할 수 있습니다. 실무에서는 초기 검증 단계에서는 stuff를, 문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다. 검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}

fetch_k
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
k
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.
일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [56]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_184/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. The narrator perceives Demian as being like an animal, a spirit, or an image, and describes him as unimaginably different from others.

마크다운 형식으로 출력

In [60]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. The narrator perceives Demian as being like an animal, a spirit, or an image, and describes him as unimaginably different from others.

RAG를 사용하지 않은 llm 호출 시도

In [58]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))

If you are referring to Emil Sinclair, the protagonist of "Demian: The Story of Emil Sinclair's Youth" by Hermann Hesse, his physical appearance is not described in detail in the book. The novel focuses more on Sinclair's inner psychological world, his thoughts, and his spiritual development rather than his physical attributes.

If you are referring to the character Max Demian, he is often portrayed as having a mysterious and captivating presence rather than specific physical traits. His character is depicted as being charismatic, insightful, and possessing a profound influence on Emil Sinclair. The novel emphasizes more on his philosophical and spiritual guidance and the aura he exudes rather than his outward appearance. 

In both cases, much of the emphasis is on the psychological and symbolic roles the characters play rather than on physical descriptions.